In [1]:
! pip install catboost
! pip install shap
! pip install polars

Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 MB 56.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 8.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Looking in indexes: https://artifactory.tcsbank.ru/artifactory/api/pypi/python-all/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 93.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 134.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 12.1 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.10 -m pip in

# Ranker

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRanker, Pool
import shap
from typing import Any
import polars as pl

In [4]:
train_df = pd.read_parquet('data/processed/train_l2r.parquet')
val_df = pd.read_parquet('data/processed/val_l2r.parquet')
test_df = pd.read_parquet('data/processed/test_l2r.parquet')

In [5]:
FEATURE_COLS = [c for c in test_df.columns if c not in ('user_id', 'item_id', 'label')]
FEATURE_COLS

['als_score',
 'bpr_score',
 'knn_score',
 'ease_score',
 'pop_score',
 'n_models_recommended',
 'best_rank',
 'mean_rank',
 'rank_std',
 'n_models_top50',
 'u_positive_events_last_1_days',
 'u_positive_events_last_2_days',
 'u_positive_events_last_3_days',
 'u_all_events_last_1_days',
 'u_all_events_last_2_days',
 'u_all_events_last_3_days',
 'u_avg_timespent',
 'u_n_unique_items_last_3_days',
 'u_like_rate_last_3_days',
 'u_fav_rate_last_3_days',
 'u_skip_rate_last_3_days',
 'u_positive_rate_last_3_days',
 'u_activity_trend',
 'i_positive_events_last_1_days',
 'i_positive_events_last_2_days',
 'i_positive_events_last_3_days',
 'i_unique_users_last_1_days',
 'i_unique_users_last_2_days',
 'i_unique_users_last_3_days',
 'i_avg_timespent',
 'i_like_rate_last_3_days',
 'i_fav_rate_last_3_days',
 'i_completion_rate_last_3_days',
 'i_skip_rate_last_3_days',
 'i_interactions_per_user_last_3',
 'i_user_growth',
 'i_trend',
 'i_popularity_rank_last_3_days',
 'als_score_norm',
 'bpr_score_norm

In [6]:
def make_pool(df: pd.DataFrame, feature_cols: list) -> tuple[Pool, pd.DataFrame]:
    df_s = df.sort_values("user_id")
    X = df_s[feature_cols].values.astype(np.float32)
    y = df_s["label"].astype(int).values
    return Pool(X, label=y, group_id=df_s["user_id"].values), df_s

train_pool_r, train_sorted = make_pool(train_df, FEATURE_COLS)
val_pool_r, val_sorted   = make_pool(val_df, FEATURE_COLS)

In [7]:
def precision_at_k(recs: dict, ground_truth: dict, k: int = 20) -> float:
    scores = []
    for user_id, items in recs.items():
        gt = ground_truth.get(user_id, set())  # Другой precision, чтобы считать метрику также, как на каггле 
        top = items[:k]
        scores.append(len(set(top) & gt) / k)
    return float(np.mean(scores)) if scores else 0.0


def scores_to_recs(df: pd.DataFrame, score_col: str, k: int = 20) -> dict:
    recs = {}
    for user_id, group in df.groupby("user_id"):
        recs[user_id] = group.sort_values(score_col, ascending=False).head(k)["item_id"].tolist()
    return recs


def build_gt(df: pd.DataFrame) -> dict:
    pos = df[df["label"] == 1]
    return pos.groupby("user_id")["item_id"].apply(set).to_dict()


val_gt = build_gt(val_df)
len(val_gt)

75028

## Step 2 — CatBoost Ranker (YetiRank)

In [15]:
ranker = CatBoostRanker(
    loss_function='YetiRank',
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    l2_leaf_reg=3,
    random_seed=334791,
    eval_metric='NDCG',
    early_stopping_rounds=50,
    verbose=50,
)

ranker.fit(train_pool_r, eval_set=val_pool_r)

0:	test: 0.7093868	best: 0.7093868 (0)	total: 14.5s	remaining: 4h 2m 7s
50:	test: 0.7205630	best: 0.7205630 (50)	total: 11m 18s	remaining: 3h 30m 27s
100:	test: 0.7210996	best: 0.7210996 (100)	total: 22m 14s	remaining: 3h 17m 57s
150:	test: 0.7213596	best: 0.7213735 (141)	total: 33m 5s	remaining: 3h 6m 5s
200:	test: 0.7215798	best: 0.7215798 (200)	total: 43m 57s	remaining: 2h 54m 45s
250:	test: 0.7217029	best: 0.7217029 (250)	total: 54m 51s	remaining: 2h 43m 40s
300:	test: 0.7218090	best: 0.7218193 (299)	total: 1h 5m 40s	remaining: 2h 32m 30s
350:	test: 0.7219301	best: 0.7219301 (350)	total: 1h 16m 35s	remaining: 2h 21m 37s
400:	test: 0.7219898	best: 0.7220000 (393)	total: 1h 27m 24s	remaining: 2h 10m 33s
450:	test: 0.7220677	best: 0.7220814 (449)	total: 1h 38m 15s	remaining: 1h 59m 36s
500:	test: 0.7221001	best: 0.7221014 (497)	total: 1h 49m 10s	remaining: 1h 48m 44s
550:	test: 0.7221159	best: 0.7221365 (523)	total: 2h 2s	remaining: 1h 37m 48s
600:	test: 0.7221682	best: 0.7221827 (597

CatBoostRanker(depth=6, early_stopping_rounds=50, eval_metric='NDCG', iterations=1000, l2_leaf_reg=3, learning_rate=0.1, loss_function='YetiRank', random_seed=334791, verbose=50)

In [8]:
ranker.save_model("ranker.cbm")

CatBoostRanker(depth=6, eval_metric='NDCG', iterations=1000, l2_leaf_reg=3, learning_rate=0.03, loss_function='YetiRank', od_wait=50, random_seed=334791, verbose=50)

In [10]:
ranker = CatBoostRanker()
ranker.load_model("ranker.cbm")

CatBoostRanker(depth=6, eval_metric='NDCG', iterations=1000, l2_leaf_reg=3, learning_rate=0.03, loss_function='YetiRank', od_wait=50, random_seed=334791, verbose=50)

In [11]:
val_df_r = val_df
val_df_r["ranker_score"] = ranker.predict(val_df_r[FEATURE_COLS].values.astype(np.float32))

recs_ranker = scores_to_recs(val_df_r, "ranker_score")
p20_ranker  = precision_at_k(recs_ranker, val_gt)
p20_ranker

0.007065377153271797

In [23]:
sample_size = min(5_000, len(val_df))
val_sample = val_df.sample(sample_size, random_state=42)

explainer = shap.TreeExplainer(ranker)
shap_values = explainer.shap_values(val_sample[FEATURE_COLS].values.astype(np.float32))

shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURE_COLS,
    name="mean_|shap|",
).sort_values(ascending=False)

shap_importance.head(30)

bpr_score                         0.244708
bpr_score_norm                    0.235752
als_score_norm                    0.169451
i_positive_events_last_1_days     0.143453
als_score                         0.122779
i_user_growth                     0.106543
mean_rank                         0.098730
i_unique_users_last_1_days        0.066907
ease_score_norm                   0.038868
i_trend                           0.033092
ease_score                        0.032655
i_fav_rate_last_3_days            0.029973
knn_score                         0.029503
best_rank                         0.028844
pop_score                         0.026056
i_positive_events_last_3_days     0.025965
i_positive_events_last_2_days     0.021419
n_models_recommended              0.019062
i_unique_users_last_2_days        0.015998
i_popularity_rank_last_3_days     0.014324
u_positive_rate_last_3_days       0.010596
i_like_rate_last_3_days           0.010536
u_n_unique_items_last_3_days      0.010246
i_avg_times

In [12]:
test_scores = ranker.predict(test_df[FEATURE_COLS].values.astype(np.float32))

test_df_out = test_df[["user_id", "item_id"]].copy()
test_df_out["score"] = test_scores

In [15]:
total_dataset = pl.read_parquet("data/train.parquet").to_pandas()
target_users = pl.read_parquet("data/target_user_ids.parquet").to_pandas()
target_user_ids = target_users["user_id"].values

In [16]:
test_recs = (
    test_df_out.sort_values("score", ascending=False)
               .groupby("user_id", sort=False)["item_id"]
               .apply(lambda x: str(x.head(20).tolist()))
               .reset_index()
               .rename(columns={"item_id": "item_ids"})
)
submission = target_users.merge(test_recs, on="user_id", how="left")

In [17]:
submission

,user_id,item_ids
0,16168876412835480816,"[88039, 14874, 65916, 59304, 71798, 144860, 75..."
1,641347240838295768,"[22011, 69092, 45184, 21566, 597, 28113, 81878..."
2,11282462919885101133,"[86064, 147011, 62359, 1821, 259644, 51975, 10..."
3,10187162321047637271,"[19827, 28522, 30485, 20774, 77794, 23939, 683..."
4,1522284854545669005,"[2655, 36369, 51268, 12133, 22670, 4889, 41136..."
...,...,...
200147,15721614028231690371,"[28958, 76765, 59033, 39542, 5068, 11946, 3562..."
200148,13933216566018619926,"[19355, 69857, 163255, 264083, 212594, 59417, ..."
200149,9131592974022462243,"[19566, 29582, 128707, 64759, 24003, 62886, 81..."
200150,18270889459511706192,"[58894, 71070, 70101, 151584, 32538, 70062, 17..."


In [20]:
filtered = total_dataset[(total_dataset["watch_time"] > 60) | (total_dataset["event_type"] != "watch_time")]
top_items = list(total_dataset["item_id"].value_counts()[:20].index)
top_items_set = set(top_items)
seen_all = total_dataset.groupby("user_id")["item_id"].apply(set).to_dict()

In [ ]:
stats = {"users_filled": 0, "total_pop_slots": 0, "pop_seen_removed": 0}

def fill_with_popular(row: dict[str, Any]):
    items = row["item_ids"]
    uid = row["user_id"]
    seen = seen_all.get(uid, set())

    if type(items) != float:
        return str(items)
        
    items = []
    stats["pop_seen_removed"] += len(top_items_set & seen)

    extra = [i for i in top_items if i not in seen and i not in set(items or [])]
    n_pop = min(len(extra), 20 - len(items or []))
    if n_pop > 0:
        stats["users_filled"] += 1
        stats["total_pop_slots"] += n_pop

    return str((list(items or []) + extra)[:20])

submission["item_ids"] = submission.apply(fill_with_popular, axis=1)
stats

{'users_filled': 33, 'total_pop_slots': 660, 'pop_seen_removed': 0}

In [22]:
submission.to_csv("sub5.csv", index=False)